# Stable Video Diffusion (SVD) — Image to Video

Generate short video clips from a single image using [Stable Video Diffusion](https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt) by Stability AI.

- **SVD-XT**: 25 frames at 576×1024
- **Runtime**: T4 (free tier) or A100 (faster)
- **VRAM**: ~8GB (float16)

Go to **Runtime > Change runtime type > T4 GPU** (or A100 for speed)

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors pillow imageio[ffmpeg]

import torch
print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = props.total_mem if hasattr(props, 'total_mem') else props.total_memory
    print(f'GPU: {props.name} — {vram / 1024**3:.1f} GiB')
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 or A100')
print('Dependencies installed.')

In [ ]:
# Cell 2: Load SVD-XT pipeline
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
import torch

pipe = StableVideoDiffusionPipeline.from_pretrained(
    'stabilityai/stable-video-diffusion-img2vid-xt',
    torch_dtype=torch.float16,
    variant='fp16',
)
pipe.to('cuda')

# Enable memory optimizations
pipe.enable_model_cpu_offload()
pipe.unet.enable_forward_chunking()

print('SVD-XT pipeline loaded!')

In [ ]:
# Cell 3: Load input image
# Option A: Upload your own image
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = list(uploaded.keys())[0]

# Option B: Use a URL
IMAGE_URL = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/svd/rocket.png'

from PIL import Image
import requests
from io import BytesIO

# Load and resize to SVD's expected resolution
image = load_image(IMAGE_URL)
image = image.resize((1024, 576))  # SVD-XT native resolution

print(f'Input image: {image.size}')
image

In [ ]:
# Cell 4: Generate video
import torch

generator = torch.manual_seed(42)

frames = pipe(
    image,
    decode_chunk_size=8,        # lower = less VRAM, slower
    generator=generator,
    num_frames=25,              # SVD-XT supports 25 frames
    motion_bucket_id=127,       # 1-255, higher = more motion
    noise_aug_strength=0.02,    # 0-1, how much noise to add to input
).frames[0]

print(f'Generated {len(frames)} frames')

In [ ]:
# Cell 5: Export to MP4 and display
from diffusers.utils import export_to_video
from IPython.display import HTML
from base64 import b64encode

OUTPUT_PATH = '/content/svd_output.mp4'
export_to_video(frames, OUTPUT_PATH, fps=7)
print(f'Video saved to {OUTPUT_PATH}')

# Display inline
with open(OUTPUT_PATH, 'rb') as f:
    mp4 = b64encode(f.read()).decode()
HTML(f'''
<video width="1024" controls autoplay loop>
  <source src="data:video/mp4;base64,{mp4}" type="video/mp4">
</video>
''')

In [ ]:
# Cell 6: Generate from your own image (upload)
from google.colab import files
from PIL import Image

print('Upload an image:')
uploaded = files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]
    custom_image = Image.open(img_path).resize((1024, 576))
    print(f'Loaded: {img_path} -> {custom_image.size}')
    display(custom_image)

    # Generate
    generator = torch.manual_seed(42)
    custom_frames = pipe(
        custom_image,
        decode_chunk_size=8,
        generator=generator,
        num_frames=25,
        motion_bucket_id=127,
        noise_aug_strength=0.02,
    ).frames[0]

    custom_output = f'/content/svd_{img_path.rsplit(".", 1)[0]}.mp4'
    export_to_video(custom_frames, custom_output, fps=7)
    print(f'Video saved: {custom_output}')

    with open(custom_output, 'rb') as f:
        mp4 = b64encode(f.read()).decode()
    display(HTML(f'<video width="1024" controls autoplay loop><source src="data:video/mp4;base64,{mp4}" type="video/mp4"></video>'))

In [ ]:
# Cell 7: Batch generate with different motion levels
import os

MOTION_LEVELS = [50, 127, 200]  # low, medium, high motion

for motion in MOTION_LEVELS:
    generator = torch.manual_seed(42)
    result_frames = pipe(
        image,
        decode_chunk_size=8,
        generator=generator,
        num_frames=25,
        motion_bucket_id=motion,
        noise_aug_strength=0.02,
    ).frames[0]

    out_path = f'/content/svd_motion_{motion}.mp4'
    export_to_video(result_frames, out_path, fps=7)
    print(f'Motion {motion}: saved to {out_path}')

print('\nAll variants generated!')

In [ ]:
# Cell 8: Download videos
from google.colab import files
import glob

for mp4_file in sorted(glob.glob('/content/svd_*.mp4')):
    size = os.path.getsize(mp4_file) / 1024**2
    print(f'Downloading {os.path.basename(mp4_file)} ({size:.1f} MB)...')
    files.download(mp4_file)

print('Done!')